# Модуль D — каузальные методы: наблюдательная оценка эффекта + uplift

**Идея модуля.** Hillstrom — рандомизированный эксперимент, поэтому истинный эффект (ATE) нам **известен**. Мы используем его как эталон, чтобы:

1. Зафиксировать **истинный ATE** из рандомизации (ground truth).
2. Искусственно создать **смещённую наблюдательную выборку** (конфаундинг: попадание в treatment зависит от ковариаты, которая влияет и на исход).
3. Показать, что **наивное сравнение врёт**.
4. Исправить смещение через **propensity score** (IPW) и **регрессионную корректировку** — и вернуться к истине.
5. **Uplift / CATE** — оценить неоднородность эффекта: кого выгоднее таргетить.

Это и есть «инкрементальность»: причинный эффект воздействия, а не корреляция. И проверка методов против экспериментальной истины — то, чего нельзя показать на чисто наблюдательных данных.

> Данные реальные и открытые (Hillstrom / MineThatData). Не продакшн-данные компании; методология и код — рабочие.


## 1. Зависимости и данные

In [1]:
%pip install -q scikit-learn scipy pandas numpy matplotlib
print("готово")

Note: you may need to restart the kernel to use updated packages.
готово


In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(42)

URL = "http://www.minethatdata.com/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
LOCAL = "hillstrom.csv"   # если ссылка недоступна — положи CSV рядом с ноутбуком
df = None
try:
    import os
    src = LOCAL if os.path.exists(LOCAL) else URL
    df = pd.read_csv(src); print("Загружено:", "локальный файл" if src == LOCAL else "minethatdata", df.shape)
except Exception as e:
    print("Прямая ссылка не вышла:", repr(e))
    try:
        !pip -q install scikit-uplift
        from sklift.datasets import fetch_hillstrom
        b = fetch_hillstrom(target_col='spend')
        df = b.data.copy(); df['segment']=b.treatment; df['spend']=b.target
        df['conversion']=fetch_hillstrom(target_col='conversion').target
        df['visit']=fetch_hillstrom(target_col='visit').target
        print("Загружено через scikit-uplift:", df.shape)
    except Exception as e2:
        from google.colab import files
        up=files.upload(); df=pd.read_csv(list(up.keys())[0])
print("Колонки:", list(df.columns))
df.head(3)

Загружено: локальный файл (64000, 12)
Колонки: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0


## 2. Подготовка: treatment/control, ковариаты, исход

Как в Модуле C: **treatment = Womens E-Mail**, **control = No E-Mail**.
Ковариаты (то, что известно ДО воздействия): `recency`, `history`, `mens`, `womens`, `newbie`, `channel`, `zip_code`.

In [3]:
TREATMENT, CONTROL = "Womens E-Mail", "No E-Mail"
ab = df[df["segment"].isin([TREATMENT, CONTROL])].copy()
ab["treat"] = (ab["segment"] == TREATMENT).astype(int)

# ковариаты -> числовая матрица
covs_num = ["recency","history","mens","womens","newbie"]
X = pd.concat([ab[covs_num].reset_index(drop=True),
               pd.get_dummies(ab[["channel","zip_code"]].reset_index(drop=True), drop_first=True).astype(int)],
              axis=1)
ab = ab.reset_index(drop=True)
y_conv = ab["conversion"].values
print("Ковариат:", X.shape[1], "| групп:", dict(ab["treat"].value_counts()))
X.head(3)

Ковариат: 9 | групп: {1: np.int64(21387), 0: np.int64(21306)}


,recency,history,mens,womens,newbie,channel_Phone,channel_Web,zip_code_Surburban,zip_code_Urban
0,10,142.44,1,0,0,1,0,1,0
1,6,329.08,1,1,1,0,1,0,0
2,7,180.65,0,1,1,0,1,1,0


## 3. Экспериментальный ATE (ground truth)
На рандомизированных данных простая разница средних = несмещённый причинный эффект. Здесь исход — **visit** (частый, ~10–15%): на нём каузальные оценки устойчивее, чем на редкой конверсии.

In [4]:
ate_exp = ab.loc[ab.treat==1,"visit"].mean() - ab.loc[ab.treat==0,"visit"].mean()
print(f"Экспериментальный ATE по visit = {ate_exp:+.4f}  <-- ЭТАЛОН")

Экспериментальный ATE по visit = +0.0452  <-- ЭТАЛОН


## 4. Создаём смещённую наблюдательную выборку (конфаундинг)

Имитируем ситуацию «данные не из эксперимента»: в выборку treated-клиентов с высоким прошлым тратами (`history`) берём чаще, а control с высоким `history` — реже. Тогда treatment коррелирует с `history`, а `history` влияет на конверсию → классический конфаундинг.

In [5]:
med = ab["history"].median()
keep_t = rng.random(len(ab)) < (0.4 + 0.5*(ab["history"] >  med))
keep_c = rng.random(len(ab)) < (0.4 + 0.5*(ab["history"] <= med))
mask = ((ab.treat==1) & keep_t) | ((ab.treat==0) & keep_c)
obs = ab[mask].reset_index(drop=True)
Xo  = X[mask.values].reset_index(drop=True)

print("Дисбаланс ковариаты history (среднее):")
print(f"  в эксперименте:  treat={ab[ab.treat==1].history.mean():.0f}, control={ab[ab.treat==0].history.mean():.0f}")
print(f"  в набл.выборке:  treat={obs[obs.treat==1].history.mean():.0f}, control={obs[obs.treat==0].history.mean():.0f}  <-- перекос")

ate_naive = obs.loc[obs.treat==1,"visit"].mean() - obs.loc[obs.treat==0,"visit"].mean()
print(f"\nНаивная оценка на смещённой выборке = {ate_naive:+.4f}")
print(f"Экспериментальная истина            = {ate_exp:+.4f}")
print(f"-> наивная оценка смещена на {ate_naive-ate_exp:+.4f}")

Дисбаланс ковариаты history (среднее):
  в эксперименте:  treat=243, control=241
  в набл.выборке:  treat=307, control=176  <-- перекос

Наивная оценка на смещённой выборке = +0.0591
Экспериментальная истина            = +0.0452
-> наивная оценка смещена на +0.0138


## 5. Коррекция смещения: propensity score

**Propensity score** = вероятность попасть в treatment по ковариатам. Через неё выравниваем группы двумя способами: **IPW** (взвешивание) и **регрессионная корректировка** (S-learner).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# propensity
ps = LogisticRegression(max_iter=1000).fit(Xo, obs["treat"]).predict_proba(Xo)[:,1]
ps = np.clip(ps, 0.02, 0.98)

# IPW
w1 = obs.treat.values / ps
w0 = (1-obs.treat.values) / (1-ps)
ate_ipw = np.sum(w1*obs.visit)/np.sum(w1) - np.sum(w0*obs.visit)/np.sum(w0)

# регрессионная корректировка (S-learner)
Xt = Xo.assign(treat=1); Xc = Xo.assign(treat=0)
m = GradientBoostingClassifier(max_depth=3, n_estimators=200).fit(Xo.assign(treat=obs.treat), obs.visit)
ate_reg = (m.predict_proba(Xt)[:,1] - m.predict_proba(Xc)[:,1]).mean()

print(f"Наивная оценка          = {ate_naive:+.4f}")
print(f"IPW-коррекция           = {ate_ipw:+.4f}")
print(f"Регрессионная коррекция = {ate_reg:+.4f}")
print(f"Экспериментальная истина= {ate_exp:+.4f}  <-- эталон")
print(f"\nСнижение смещения: наивная ошибалась на {abs(ate_naive-ate_exp):.4f}, "
      f"IPW — на {abs(ate_ipw-ate_exp):.4f}, регрессия — на {abs(ate_reg-ate_exp):.4f}")

Наивная оценка          = +0.0591
IPW-коррекция           = +0.0347
Регрессионная коррекция = +0.0398
Экспериментальная истина= +0.0452  <-- эталон

Снижение смещения: наивная ошибалась на 0.0138, IPW — на 0.0106, регрессия — на 0.0054


## 6. Uplift / CATE — кого таргетить

Эффект неоднороден: одним письмо помогает сильно, другим — почти нет. **Uplift-модель** предсказывает индивидуальный прирост. Оцениваем на рандомизированных данных, исход — **visit** (сигнала больше, чем у редкой конверсии).

Метод — **T-learner**: две модели (на treatment и control), uplift = разница вероятностей.

In [7]:
tr = ab[ab.treat==1]; ct = ab[ab.treat==0]
Xtr = X[ab.treat.values==1]; Xct = X[ab.treat.values==0]

mt = GradientBoostingClassifier(max_depth=3, n_estimators=200).fit(Xtr, tr["visit"])
mc = GradientBoostingClassifier(max_depth=3, n_estimators=200).fit(Xct, ct["visit"])

ab["uplift"] = mt.predict_proba(X)[:,1] - mc.predict_proba(X)[:,1]
# децили по предсказанному аплифту (rank -> без проблем с дублями границ)
ab["decile"] = pd.qcut(ab["uplift"].rank(method="first"), 10, labels=False)

# фактический аплифт по децилям = валидация модели
rows=[]
for d,g in ab.groupby("decile"):
    up = g[g.treat==1]["visit"].mean() - g[g.treat==0]["visit"].mean()
    rows.append((int(d), g["uplift"].mean(), up))
res = pd.DataFrame(rows, columns=["дециль","предсказ_аплифт","фактич_аплифт"])
print(res.to_string(index=False))
print(f"\nВерхний дециль: фактический visit-аплифт = {res.iloc[-1]['фактич_аплифт']:+.4f}")
print(f"Нижний дециль:  фактический visit-аплифт = {res.iloc[0]['фактич_аплифт']:+.4f}")
print("Если фактический аплифт растёт от нижнего дециля к верхнему — модель верно ранжирует отклик.")

 дециль  предсказ_аплифт  фактич_аплифт
      0        -0.044269      -0.144717
      1         0.001540      -0.029079
      2         0.014134      -0.020685
      3         0.023884       0.024339
      4         0.035672       0.035828
      5         0.052030       0.039690
      6         0.066042       0.054810
      7         0.077269       0.090723
      8         0.090668       0.122746
      9         0.137505       0.279739

Верхний дециль: фактический visit-аплифт = +0.2797
Нижний дециль:  фактический visit-аплифт = -0.1447
Если фактический аплифт растёт от нижнего дециля к верхнему — модель верно ранжирует отклик.


## 7. Выводы

- **Эталон:** экспериментальный ATE по visit = **+0.0452** (несмещённый, из рандомизации).
- **Конфаундинг:** на смещённой наблюдательной выборке наивная оценка = **+0.0591** (завышена на +0.0138 — перекос по `history`: 307 vs 176).
- **Коррекция:** IPW = **+0.0347** (ошибка 0.0106), регрессионная корректировка = **+0.0398** (ошибка 0.0054). Обе вернули оценку к истине; регрессия точнее — IPW шумит на больших весах.
- **Uplift (T-learner):** фактический visit-аплифт монотонно растёт по децилям от **−0.14** (нижний) до **+0.28** (верхний). Модель верно ранжирует отклик; нижним сегментам письмо даже вредит → таргетировать топ-децили, а не всю базу.

**Итог:** propensity-методы восстанавливают причинный эффект при известном смещении (проверено против экспериментальной истины), а uplift переводит «средний эффект» в «кого таргетить».

---
### Что показывает этот модуль
Наблюдательная каузальная оценка (propensity score: IPW + регрессионная корректировка) с валидацией против экспериментальной истины, плюс uplift/CATE для таргетинга. На реальном рандомизированном эксперименте. Прямой ответ на вопрос «умеешь ли считать инкрементальность».